<a href="https://colab.research.google.com/github/DMS-999/ML-Handbook-materials/blob/main/pdf_parser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pdfplumber pandas openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 22.0 MB/s eta 0:00:00


In [4]:
import pdfplumber
import pandas as pd
import os
import re

def pdfplumber_to_excel(pdf_file_path, excel_file_path):
    """Извлечение таблиц с помощью pdfplumber (без Java)"""

    if not os.path.exists(pdf_file_path):
        print(f" Файл '{pdf_file_path}' не найден.")
        return

    all_tables = []

    with pdfplumber.open(pdf_file_path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            # Извлекаем таблицы со страницы
            tables = page.extract_tables()

            for table_num, table in enumerate(tables):
                if table and len(table) > 1:  # проверка что таблица не пустая
                    # Конвертируем в DataFrame
                    df = pd.DataFrame(table[1:], columns=table[0] if table[0] else None)
                    all_tables.append((page_num, table_num, df))
                    print(f"   Страница {page_num}, Таблица {table_num + 1}: {df.shape}")

    if not all_tables:
        print(" Таблицы не найдены.")
        return

    # Сохраняем в Excel
    with pd.ExcelWriter(excel_file_path, engine='openpyxl') as writer:
        for page_num, table_num, df in all_tables:
            sheet_name = f"Page{page_num}_Table{table_num+1}"
            sheet_name = re.sub(r'[\\/*?:\[\]]', '_', sheet_name)[:31]
            df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"✅ Сохранено {len(all_tables)} таблиц в {excel_file_path}")

# Использование
pdfplumber_to_excel("/content/sample-tables.pdf", "output.xlsx")

   Страница 1, Таблица 1: (2, 3)
   Страница 1, Таблица 2: (7, 4)
   Страница 1, Таблица 3: (5, 2)
   Страница 2, Таблица 1: (6, 2)
   Страница 2, Таблица 2: (8, 4)
   Страница 2, Таблица 3: (8, 5)
   Страница 3, Таблица 1: (3, 4)
   Страница 3, Таблица 2: (3, 4)
   Страница 3, Таблица 3: (3, 5)
   Страница 4, Таблица 1: (13, 5)
   Страница 4, Таблица 2: (11, 4)
   Страница 5, Таблица 1: (4, 5)
   Страница 5, Таблица 2: (3, 4)
   Страница 5, Таблица 3: (3, 4)
   Страница 5, Таблица 4: (6, 5)
   Страница 6, Таблица 1: (6, 5)
   Страница 6, Таблица 2: (10, 3)
   Страница 6, Таблица 3: (10, 3)
   Страница 7, Таблица 1: (5, 5)
   Страница 7, Таблица 2: (7, 4)
   Страница 8, Таблица 1: (7, 4)
   Страница 8, Таблица 2: (7, 4)
   Страница 8, Таблица 3: (3, 5)
   Страница 9, Таблица 1: (12, 4)
   Страница 9, Таблица 2: (4, 5)
   Страница 9, Таблица 3: (6, 5)
   Страница 10, Таблица 1: (3, 3)
   Страница 10, Таблица 2: (12, 4)
   Страница 11, Таблица 1: (12, 5)
✅ Сохранено 29 таблиц в output.xl